In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import pandas as pd
ds=pd.read_csv('/kaggle/input/datasets/lakshyaupadhyaya/movie-with-sentiments/movie_columns_with_sentiments.csv')

In [2]:
ds.drop(columns='embedding_text',inplace=True)

In [5]:
ds.columns

Index(['title', 'description', 'directed_by', 'produced_by', 'starring',
       'country', 'language', 'release_date', 'emotion_analysis',
       'main_sentiment', 'main_sentiment_score', 'all_sentiments'],
      dtype='object')

In [3]:
def create_embedding_text(row):
    parts = []
    
    if pd.notnull(row['title']):
        parts.append(f"Title: {row['title']}")
    
    if pd.notnull(row['description']):
        parts.append(f"Description: {row['description']}")
    
    if pd.notnull(row['directed_by']):
        parts.append(f"Directed by: {row['directed_by']}")
    
    if pd.notnull(row['starring']):
        parts.append(f"Starring: {row['starring']}")
    
    if pd.notnull(row['main_sentiment']):
        parts.append(f"Emotion: {row['main_sentiment']}")
    
    return " | ".join(parts)

ds['embedding_text'] = ds.apply(create_embedding_text, axis=1)

In [5]:
ds.to_csv('movie_columns_with_sentiment.csv', index=False)

In [10]:
!pip install supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 11.1 MB/s eta 0:00:00


In [11]:
from supabase import create_client
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
url = secrets.get_secret("SUPABASE_URL")      # https://ezfdflffkszhyumqcvoz.supabase.co
key = secrets.get_secret("SUPABASE_KEY")      # your anon key from Settings → API

supabase = create_client(url, key)

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    device="cuda",
    trust_remote_code=True
)

for row in ds.itertuples(index=True, name='Pandas'):
    embedding_text = row.embedding_text
    vector_embedding = model.encode([embedding_text])[0].tolist()
    
    response = (
        supabase.table('movies')
        .insert({
            "title": row.title,
            "description": row.description,
            "directed_by": row.directed_by,
            "produced_by": row.produced_by,
            "starring": row.starring,
            "country": row.country,
            "language": row.language,
            "release_date": str(row.release_date),
            "main_sentiment": row.main_sentiment,
            "main_sentiment_score": float(row.main_sentiment_score),
            "all_sentiments": row.all_sentiments,
            "embedding": vector_embedding
        })
        .execute()
    )

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

<All keys matched successfully>


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]